# 📈 국장 앙상블 주가 방향 예측 모델 v3
> **핵심 개선사항**
- 레이블 단순화 (threshold 제거)
- inf/nan 완벽 처리
- 피처 선택 강화 (상관관계 낮은 피처 제거)
- LightGBM 비중 강화
- Optuna 하이퍼파라미터 자동 튜닝

## 📦 1. 설치

In [ ]:
!pip install pykrx ta scikit-learn lightgbm yfinance optuna matplotlib seaborn -q
!apt-get install -y fonts-nanum -qq

## 📚 2. 임포트

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import warnings, pickle
warnings.filterwarnings('ignore')

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
font_prop = fm.FontProperties(fname=font_path)
plt.rcParams['font.family'] = font_prop.get_name()
plt.rcParams['axes.unicode_minus'] = False

from pykrx import stock
import yfinance as yf
from datetime import datetime, timedelta

from ta.momentum import RSIIndicator, StochasticOscillator
from ta.trend import MACD, EMAIndicator, SMAIndicator
from ta.volatility import BollingerBands, AverageTrueRange
from ta.volume import OnBalanceVolumeIndicator

from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, f1_score, roc_curve)
from sklearn.utils.class_weight import compute_class_weight
import lightgbm as lgb

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, LSTM, Bidirectional, Dense, Dropout,
    BatchNormalization, Activation, Permute, Multiply, GlobalAveragePooling1D)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

print(f'✅ TensorFlow : {tf.__version__}')
print(f'✅ LightGBM   : {lgb.__version__}')
print(f'✅ Optuna     : {optuna.__version__}')
print(f'✅ GPU        : {tf.config.list_physical_devices("GPU")}')

## ⚙️ 3. 설정값

In [ ]:
# ===================================================
# 🎯 종목코드 입력
# 069500: KOSPI ETF / 005930: 삼성전자 / 000660: SK하이닉스
# ===================================================
TICKER     = '005930'
START_DATE = '20150101'
END_DATE   = datetime.today().strftime('%Y%m%d')

SEQ_LEN       = 60
PRED_DAYS     = 10     # 10일로 줄임 (20일은 너무 노이즈 많음)
BATCH_SIZE    = 64
EPOCHS        = 150
LSTM_UNITS    = [128, 64]
DROPOUT       = 0.3
LEARNING_RATE = 0.001
TEST_RATIO    = 0.2
VAL_RATIO     = 0.1

# 앙상블 가중치 (LightGBM 중심)
W_LSTM = 0.3
W_LGBM = 0.5
W_RF   = 0.2

print(f'📌 종목코드  : {TICKER}')
print(f'📅 기간      : {START_DATE} ~ {END_DATE}')
print(f'🔮 예측기간  : {PRED_DAYS}일 후')
print(f'⚖️  앙상블    : LSTM {W_LSTM} / LightGBM {W_LGBM} / RF {W_RF}')

## 📥 4. 데이터 수집

In [ ]:
def fetch_krx_data(ticker, start, end):
    print(f'📡 {ticker} KRX 데이터 수집 중...')
    name = stock.get_market_ticker_name(ticker)
    print(f'  📌 종목명: {name}')
    df = stock.get_market_ohlcv_by_date(start, end, ticker)
    print(f'  📋 원본 컬럼: {list(df.columns)}')
    df = df.iloc[:, :5]
    df.columns = ['Open','High','Low','Close','Volume']
    df.index = pd.to_datetime(df.index)
    try:
        trade = stock.get_market_trading_volume_by_date(start, end, ticker)
        if '외국인합계' in trade.columns:
            df['Foreign_Buy'] = trade['외국인합계']
        if '기관합계' in trade.columns:
            df['Inst_Buy'] = trade['기관합계']
    except:
        df['Foreign_Buy'] = 0
        df['Inst_Buy']    = 0
    df.dropna(inplace=True)
    print(f'  ✅ {len(df)}일 데이터 수집 완료')
    return df, name

def fetch_external_data(start, end):
    print('📡 외부 데이터 수집 중...')
    start_dt = pd.to_datetime(start, format='%Y%m%d').strftime('%Y-%m-%d')
    end_dt   = pd.to_datetime(end,   format='%Y%m%d').strftime('%Y-%m-%d')
    tickers = {'SP500':'^GSPC', 'VIX':'^VIX', 'DXY':'DX-Y.NYB', 'OIL':'CL=F'}
    ext = pd.DataFrame()
    for name, sym in tickers.items():
        try:
            tmp = yf.download(sym, start=start_dt, end=end_dt, progress=False)['Close']
            ext[name] = tmp
            print(f'  ✅ {name}')
        except:
            print(f'  ⚠️  {name} 실패')
    ext.index = pd.to_datetime(ext.index)
    if isinstance(ext.columns, pd.MultiIndex):
        ext.columns = ext.columns.get_level_values(0)
    for col in list(ext.columns):
        ext[f'{col}_ret']  = ext[col].pct_change()
        ext[f'{col}_MA5']  = ext[col].pct_change().rolling(5).mean()
    ext.dropna(inplace=True)
    return ext

df_raw, stock_name = fetch_krx_data(TICKER, START_DATE, END_DATE)
df_ext = fetch_external_data(START_DATE, END_DATE)

df_merged = df_raw.join(df_ext, how='left')
df_merged.ffill(inplace=True)
df_merged.dropna(inplace=True)
print(f'\n✅ 병합 완료: {len(df_merged)}일 / {df_merged.shape[1]}개 컬럼')

## 🔬 5. 피처 엔지니어링

In [ ]:
def add_features(df):
    d = df.copy()

    # 가격 파생
    d['Return']     = d['Close'].pct_change()
    d['Return_2d']  = d['Close'].pct_change(2)
    d['Return_5d']  = d['Close'].pct_change(5)
    d['Return_10d'] = d['Close'].pct_change(10)
    d['Return_20d'] = d['Close'].pct_change(20)
    d['HL_ratio']   = (d['High'] - d['Low']) / d['Close']
    d['OC_ratio']   = (d['Close'] - d['Open']) / (d['Open'] + 1e-9)
    d['Gap']        = (d['Open'] - d['Close'].shift(1)) / (d['Close'].shift(1) + 1e-9)

    # 이동평균 & 괴리율
    for w in [5, 10, 20, 60, 120]:
        sma = SMAIndicator(d['Close'], window=w).sma_indicator()
        d[f'SMA_{w}']      = sma
        d[f'SMA_dist_{w}'] = (d['Close'] - sma) / (sma + 1e-9)
    for w in [12, 26]:
        d[f'EMA_{w}'] = EMAIndicator(d['Close'], window=w).ema_indicator()

    # MACD
    macd = MACD(d['Close'])
    d['MACD']        = macd.macd()
    d['MACD_signal'] = macd.macd_signal()
    d['MACD_diff']   = macd.macd_diff()

    # RSI
    for w in [7, 14, 21]:
        d[f'RSI_{w}'] = RSIIndicator(d['Close'], window=w).rsi()

    # Stochastic
    stoch = StochasticOscillator(d['High'], d['Low'], d['Close'])
    d['Stoch_k'] = stoch.stoch()
    d['Stoch_d'] = stoch.stoch_signal()

    # Bollinger Bands
    bb = BollingerBands(d['Close'])
    d['BB_width'] = (bb.bollinger_hband() - bb.bollinger_lband()) / (bb.bollinger_mavg() + 1e-9)
    d['BB_pct']   = bb.bollinger_pband()

    # ATR
    for w in [7, 14]:
        d[f'ATR_{w}'] = AverageTrueRange(d['High'], d['Low'], d['Close'], window=w).average_true_range()
        d[f'ATR_{w}_norm'] = d[f'ATR_{w}'] / (d['Close'] + 1e-9)

    # 거래량
    vol_ma20 = d['Volume'].rolling(20).mean()
    d['Volume_ratio'] = d['Volume'] / (vol_ma20 + 1e-9)
    d['OBV']          = OnBalanceVolumeIndicator(d['Close'], d['Volume']).on_balance_volume()
    d['OBV_MA']       = d['OBV'].rolling(10).mean()
    d['OBV_dist']     = (d['OBV'] - d['OBV_MA']) / (d['OBV_MA'].abs() + 1e-9)

    # 변동성
    d['Vol_5d']  = d['Return'].rolling(5).std()
    d['Vol_20d'] = d['Return'].rolling(20).std()
    d['Vol_ratio'] = d['Vol_5d'] / (d['Vol_20d'] + 1e-9)

    # 외국인/기관
    if 'Foreign_Buy' in d.columns:
        d['Foreign_MA5']  = d['Foreign_Buy'].rolling(5).mean()
        d['Foreign_MA20'] = d['Foreign_Buy'].rolling(20).mean()
    if 'Inst_Buy' in d.columns:
        d['Inst_MA5']  = d['Inst_Buy'].rolling(5).mean()
        d['Inst_MA20'] = d['Inst_Buy'].rolling(20).mean()

    # ✅ 레이블: 단순 상승/하락 (threshold 없음)
    d['Label'] = (d['Close'].shift(-PRED_DAYS) > d['Close']).astype(int)

    d.dropna(inplace=True)

    # ✅ inf 완벽 처리
    d.replace([np.inf, -np.inf], np.nan, inplace=True)
    d.ffill(inplace=True)
    d.bfill(inplace=True)
    d.dropna(inplace=True)

    return d

df_feat = add_features(df_merged)
vc = df_feat['Label'].value_counts()
print(f'✅ 피처 수: {df_feat.shape[1]} | 샘플 수: {len(df_feat)}')
print(f'  📊 상승(1): {vc.get(1,0)} ({vc.get(1,0)/len(df_feat)*100:.1f}%)')
print(f'  📊 하락(0): {vc.get(0,0)} ({vc.get(0,0)/len(df_feat)*100:.1f}%)')

## 🔧 6. 전처리 & 피처 선택 & 시퀀스 생성

In [ ]:
exclude = ['Label','Open','High','Low','Close','Volume',
           'Foreign_Buy','Inst_Buy','SP500','VIX','DXY','OIL']
feature_cols = [c for c in df_feat.columns if c not in exclude]

X_raw = df_feat[feature_cols].values
y_raw = df_feat['Label'].values

# inf/nan 최종 체크
X_raw = np.where(np.isinf(X_raw), np.nan, X_raw)
df_temp = pd.DataFrame(X_raw, columns=feature_cols)
df_temp.ffill(inplace=True)
df_temp.bfill(inplace=True)
X_raw = df_temp.values

print(f'🔢 inf 체크: {np.isinf(X_raw).sum()} | nan 체크: {np.isnan(X_raw).sum()}')

n          = len(X_raw)
test_size  = int(n * TEST_RATIO)
val_size   = int(n * VAL_RATIO)
train_size = n - test_size - val_size

scaler = RobustScaler()
X_tr_sc = scaler.fit_transform(X_raw[:train_size])
X_vl_sc = scaler.transform(X_raw[train_size:train_size+val_size])
X_te_sc = scaler.transform(X_raw[train_size+val_size:])

y_train = y_raw[:train_size]
y_val   = y_raw[train_size:train_size+val_size]
y_test  = y_raw[train_size+val_size:]

# ✅ 피처 선택 (RF로 중요도 낮은 피처 제거)
print('\n🌳 피처 선택 중...')
rf_selector = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_selector.fit(X_tr_sc, y_train)
selector = SelectFromModel(rf_selector, threshold='mean', prefit=True)
X_tr_sel = selector.transform(X_tr_sc)
X_vl_sel = selector.transform(X_vl_sc)
X_te_sel = selector.transform(X_te_sc)

selected_cols = [feature_cols[i] for i in range(len(feature_cols)) if selector.get_support()[i]]
print(f'  피처 선택: {len(feature_cols)}개 → {len(selected_cols)}개')
print(f'  선택된 피처: {selected_cols}')

def make_sequences(X, y, seq_len):
    xs, ys = [], []
    for i in range(seq_len, len(X)):
        xs.append(X[i-seq_len:i])
        ys.append(y[i])
    return np.array(xs), np.array(ys)

X_tr, y_tr = make_sequences(X_tr_sel, y_train, SEQ_LEN)
X_vl, y_vl = make_sequences(X_vl_sel, y_val,   SEQ_LEN)
X_te, y_te = make_sequences(X_te_sel, y_test,  SEQ_LEN)

X_tr_flat = X_tr[:, -1, :]
X_vl_flat = X_vl[:, -1, :]
X_te_flat = X_te[:, -1, :]

print(f'\n📐 LSTM  - Train: {X_tr.shape} | Val: {X_vl.shape} | Test: {X_te.shape}')
print(f'📐 Flat  - Train: {X_tr_flat.shape} | Val: {X_vl_flat.shape} | Test: {X_te_flat.shape}')

## 🤖 7. BiLSTM + Attention 모델

In [ ]:
def build_lstm_model(seq_len, n_features):
    inp = Input(shape=(seq_len, n_features))
    x = Bidirectional(LSTM(LSTM_UNITS[0], return_sequences=True,
                           kernel_regularizer=l2(1e-4),
                           dropout=DROPOUT, recurrent_dropout=0.1))(inp)
    x = BatchNormalization()(x)
    x = Bidirectional(LSTM(LSTM_UNITS[1], return_sequences=True,
                           kernel_regularizer=l2(1e-4),
                           dropout=DROPOUT, recurrent_dropout=0.1))(x)
    x = BatchNormalization()(x)

    # Attention (Keras 3 호환)
    score   = Dense(1, activation='tanh')(x)
    score   = Permute((2, 1))(score)
    weights = Activation('softmax')(score)
    weights = Permute((2, 1))(weights)
    context = Multiply()([x, weights])
    context = GlobalAveragePooling1D()(context)

    x   = Dense(64, activation='relu', kernel_regularizer=l2(1e-4))(context)
    x   = Dropout(DROPOUT)(x)
    x   = Dense(32, activation='relu')(x)
    x   = Dropout(0.2)(x)
    out = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )
    return model

lstm_model = build_lstm_model(SEQ_LEN, X_tr.shape[2])
lstm_model.summary()

## 🏋️ 8. LSTM 학습

In [ ]:
cw = compute_class_weight('balanced', classes=np.array([0,1]), y=y_tr)
class_weight = {0: cw[0], 1: cw[1]}
print(f'⚖️  클래스 가중치: {class_weight}')

callbacks = [
    EarlyStopping(monitor='val_auc', patience=20,
                  restore_best_weights=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_auc', factor=0.5,
                      patience=8, min_lr=1e-6, mode='max', verbose=1),
    ModelCheckpoint('/content/best_lstm.h5', monitor='val_auc',
                    save_best_only=True, mode='max', verbose=0)
]

history = lstm_model.fit(
    X_tr, y_tr,
    validation_data=(X_vl, y_vl),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)
print('\n✅ LSTM 학습 완료!')

## 🌲 9. LightGBM + Optuna 자동 튜닝

In [ ]:
print('🔍 Optuna로 LightGBM 하이퍼파라미터 탐색 중... (약 3~5분)')

def objective(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 300, 1500),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth'        : trial.suggest_int('max_depth', 4, 10),
        'num_leaves'       : trial.suggest_int('num_leaves', 20, 100),
        'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'class_weight'     : 'balanced',
        'random_state'     : 42,
        'verbose'          : -1
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_tr_flat, y_tr,
              eval_set=[(X_vl_flat, y_vl)],
              callbacks=[lgb.early_stopping(30, verbose=False)])
    prob = model.predict_proba(X_vl_flat)[:, 1]
    return roc_auc_score(y_vl, prob)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f'\n✅ 최적 파라미터: {study.best_params}')
print(f'✅ 최적 AUC: {study.best_value:.4f}')

lgbm_model = lgb.LGBMClassifier(**study.best_params, class_weight='balanced',
                                   random_state=42, verbose=-1)
lgbm_model.fit(X_tr_flat, y_tr,
               eval_set=[(X_vl_flat, y_vl)],
               callbacks=[lgb.early_stopping(50, verbose=False),
                          lgb.log_evaluation(100)])
print(f'✅ LightGBM 최종 학습 완료!')

# 피처 중요도
imp = pd.Series(lgbm_model.feature_importances_, index=selected_cols)
top15 = imp.nlargest(15)
fig, ax = plt.subplots(figsize=(10, 5))
top15.sort_values().plot(kind='barh', ax=ax, color='#2196F3')
ax.set_title('LightGBM 피처 중요도 Top 15', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/feature_importance.png', dpi=150)
plt.show()

## 🌳 10. RandomForest 학습

In [ ]:
print('🌳 RandomForest 학습 중...')
rf_model = RandomForestClassifier(
    n_estimators      = 500,
    max_depth         = 10,
    min_samples_split = 10,
    min_samples_leaf  = 5,
    class_weight      = 'balanced',
    random_state      = 42,
    n_jobs            = -1
)
rf_model.fit(X_tr_flat, y_tr)
print('✅ RandomForest 완료!')

## 🎯 11. 앙상블 (Soft Voting)

In [ ]:
print('🎯 앙상블 예측 중...')

lstm_prob = lstm_model.predict(X_te, verbose=0).flatten()
lgbm_prob = lgbm_model.predict_proba(X_te_flat)[:, 1]
rf_prob   = rf_model.predict_proba(X_te_flat)[:, 1]

ensemble_prob = W_LSTM * lstm_prob + W_LGBM * lgbm_prob + W_RF * rf_prob
ensemble_pred = (ensemble_prob >= 0.5).astype(int)

lstm_pred = (lstm_prob >= 0.5).astype(int)
lgbm_pred = (lgbm_prob >= 0.5).astype(int)
rf_pred   = (rf_prob   >= 0.5).astype(int)

print('\n' + '='*55)
print(f'  📋 [{stock_name}] 모델별 테스트 성능 비교')
print('='*55)
for name, pred, prob in [
    ('LSTM        ', lstm_pred, lstm_prob),
    ('LightGBM    ', lgbm_pred, lgbm_prob),
    ('RandomForest', rf_pred,   rf_prob),
    ('🏆 앙상블   ', ensemble_pred, ensemble_prob),
]:
    acc = accuracy_score(y_te, pred)
    auc = roc_auc_score(y_te, prob)
    f1  = f1_score(y_te, pred, zero_division=0)
    print(f'  {name} | Acc={acc:.4f} | AUC={auc:.4f} | F1={f1:.4f}')
print('='*55)

## 📊 12. 시각화

In [ ]:
print(classification_report(y_te, ensemble_pred, target_names=['하락','상승']))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'{stock_name}({TICKER}) 앙상블 평가 v3', fontsize=14, fontweight='bold')

cm = confusion_matrix(y_te, ensemble_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['하락','상승'], yticklabels=['하락','상승'], ax=axes[0][0])
axes[0][0].set_title('혼동 행렬'); axes[0][0].set_ylabel('실제'); axes[0][0].set_xlabel('예측')

ax = axes[0][1]
for name, prob, color in [
    ('LSTM',        lstm_prob,     '#2196F3'),
    ('LightGBM',    lgbm_prob,     '#4CAF50'),
    ('RandomForest',rf_prob,       '#FF9800'),
    ('앙상블',      ensemble_prob, '#E91E63'),
]:
    fpr, tpr, _ = roc_curve(y_te, prob)
    auc = roc_auc_score(y_te, prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--', alpha=0.4)
ax.set_title('ROC Curve 비교'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1][0]
ax.plot(history.history['auc'],     label='Train AUC', lw=2)
ax.plot(history.history['val_auc'], label='Val AUC',   lw=2, linestyle='--')
ax.set_title('LSTM 학습 곡선'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1][1]
ax.hist(ensemble_prob[y_te==1], bins=30, alpha=0.6, color='#2196F3', label='실제 상승')
ax.hist(ensemble_prob[y_te==0], bins=30, alpha=0.6, color='#F44336', label='실제 하락')
ax.axvline(0.5, color='black', linestyle='--', lw=1.5)
ax.set_title('앙상블 확률 분포'); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/ensemble_eval_v3.png', dpi=150)
plt.show()

## 🔄 13. Walk-Forward Validation

In [ ]:
def walk_forward_eval(X_seq, X_flat, y_all, n_splits=5):
    fold_size = len(X_seq) // (n_splits + 1)
    results = []
    print('🔄 Walk-Forward Validation 시작...')
    for fold in range(n_splits):
        train_end = fold_size * (fold + 1)
        test_end  = min(train_end + fold_size, len(X_seq))
        if test_end - train_end < 10:
            continue
        Xtr_s = X_seq[:train_end];  ytr = y_all[:train_end]
        Xte_s = X_seq[train_end:test_end]; yte = y_all[train_end:test_end]
        Xtr_f = X_flat[:train_end]
        Xte_f = X_flat[train_end:test_end]

        m = build_lstm_model(SEQ_LEN, Xtr_s.shape[2])
        m.fit(Xtr_s, ytr, validation_split=0.1, epochs=40, batch_size=64,
              callbacks=[EarlyStopping(patience=8, restore_best_weights=True,
                                       monitor='val_auc', mode='max')], verbose=0)
        lp = m.predict(Xte_s, verbose=0).flatten()

        lg = lgb.LGBMClassifier(**study.best_params, class_weight='balanced',
                                  random_state=42, verbose=-1)
        lg.fit(Xtr_f, ytr)
        gp = lg.predict_proba(Xte_f)[:, 1]

        rf = RandomForestClassifier(n_estimators=200, max_depth=8,
                                     class_weight='balanced', random_state=42, n_jobs=-1)
        rf.fit(Xtr_f, ytr)
        rp = rf.predict_proba(Xte_f)[:, 1]

        ep   = W_LSTM*lp + W_LGBM*gp + W_RF*rp
        pred = (ep >= 0.5).astype(int)
        acc  = accuracy_score(yte, pred)
        f1   = f1_score(yte, pred, zero_division=0)
        auc  = roc_auc_score(yte, ep)
        results.append({'fold': fold+1, 'acc': acc, 'f1': f1, 'auc': auc})
        print(f'  Fold {fold+1}: Acc={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}')

    res_df = pd.DataFrame(results)
    print(f'\n📊 평균 Acc: {res_df.acc.mean():.4f} ± {res_df.acc.std():.4f}')
    print(f'📊 평균 F1 : {res_df.f1.mean():.4f} ± {res_df.f1.std():.4f}')
    print(f'📊 평균 AUC: {res_df.auc.mean():.4f} ± {res_df.auc.std():.4f}')
    return res_df

X_all_seq  = np.concatenate([X_tr, X_vl, X_te])
X_all_flat = np.concatenate([X_tr_flat, X_vl_flat, X_te_flat])
y_all_seq  = np.concatenate([y_tr, y_vl, y_te])
wf_results = walk_forward_eval(X_all_seq, X_all_flat, y_all_seq, n_splits=5)

## 🚀 14. 실전 예측

In [ ]:
def predict_latest(ticker=TICKER):
    df_new, nm = fetch_krx_data(ticker, START_DATE, END_DATE)
    df_ext_new = fetch_external_data(START_DATE, END_DATE)
    df_m = df_new.join(df_ext_new, how='left')
    df_m.ffill(inplace=True); df_m.dropna(inplace=True)
    df_f = add_features(df_m)

    X_new = df_f[feature_cols].values
    X_new = np.where(np.isinf(X_new), np.nan, X_new)
    df_tmp = pd.DataFrame(X_new, columns=feature_cols)
    df_tmp.ffill(inplace=True); df_tmp.bfill(inplace=True)
    X_new = df_tmp.values

    X_new_sc  = scaler.transform(X_new)
    X_new_sel = selector.transform(X_new_sc)

    seq  = X_new_sel[-SEQ_LEN:][np.newaxis, ...]
    flat = X_new_sel[-1:, :]

    lp   = lstm_model.predict(seq, verbose=0)[0][0]
    gp   = lgbm_model.predict_proba(flat)[0][1]
    rp   = rf_model.predict_proba(flat)[0][1]
    prob = W_LSTM*lp + W_LGBM*gp + W_RF*rp

    direction  = '📈 상승' if prob >= 0.5 else '📉 하락'
    conf       = max(prob, 1-prob) * 100
    last_close = int(df_f['Close'].iloc[-1])
    last_date  = df_f.index[-1].strftime('%Y-%m-%d')

    print('='*55)
    print(f'  🎯 앙상블 실전 예측: {nm}({ticker})')
    print('='*55)
    print(f'  기준일      : {last_date}')
    print(f'  현재가      : {last_close:,}원')
    print(f'  예측기간    : {PRED_DAYS}영업일 후')
    print(f'  LSTM        : {lp*100:.1f}%')
    print(f'  LightGBM    : {gp*100:.1f}%')
    print(f'  RandomForest: {rp*100:.1f}%')
    print(f'  ───────────────────────')
    print(f'  🏆 앙상블   : {direction} ({prob*100:.1f}%, 신뢰도 {conf:.1f}%)')
    print('='*55)

    recent = df_f.tail(40)
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(recent.index, recent['Close'], color='#333', lw=2)
    color = '#2196F3' if prob>=0.5 else '#F44336'
    ax.scatter(recent.index[-1], recent['Close'].iloc[-1],
               color=color, s=200, zorder=5, label=f'{direction} ({prob*100:.1f}%)')
    arrow_dy = last_close * 0.03 * (1 if prob>=0.5 else -1)
    ax.annotate('', xy=(recent.index[-1], recent['Close'].iloc[-1]+arrow_dy),
                xytext=(recent.index[-1], recent['Close'].iloc[-1]),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    ax.set_title(f'{nm}({ticker}) 최근 40일 + {PRED_DAYS}일 후 앙상블 예측', fontsize=13)
    ax.set_ylabel('주가 (원)'); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/latest_ensemble_v3.png', dpi=150)
    plt.show()
    return prob

prob = predict_latest(TICKER)

## 🔍 15. 다른 종목 즉시 예측

In [ ]:
# 종목코드만 바꾸고 실행!
predict_latest('000660')  # SK하이닉스

## 💾 16. 모델 저장

In [ ]:
lstm_model.save('/content/lstm_model_v3.h5')
lgbm_model.booster_.save_model('/content/lgbm_model_v3.txt')
with open('/content/rf_model_v3.pkl',     'wb') as f: pickle.dump(rf_model,      f)
with open('/content/scaler_v3.pkl',       'wb') as f: pickle.dump(scaler,        f)
with open('/content/selector_v3.pkl',     'wb') as f: pickle.dump(selector,      f)
with open('/content/feature_cols_v3.pkl', 'wb') as f: pickle.dump(feature_cols,  f)
print('✅ 전체 모델 저장 완료! (v3)')

---
## 📋 주요 개선사항 (v2 → v3)
| 항목 | v2 | v3 |
|------|----|----|
| 레이블 | 1% 이상 상승 | 단순 상승/하락 |
| inf 처리 | 일부 | 완벽 처리 |
| 피처 선택 | 없음 | RF 기반 자동 선택 |
| LightGBM 튜닝 | 수동 | Optuna 자동 튜닝 |
| 예측기간 | 20일 | 10일 |
| LightGBM 가중치 | 0.4 | 0.5 |

## 📋 주요 종목 코드
| 종목명 | 코드 | 종목명 | 코드 |
|--------|------|--------|------|
| KOSPI ETF | 069500 | 삼성전자 | 005930 |
| SK하이닉스 | 000660 | 카카오 | 035720 |
| NAVER | 035420 | 현대차 | 005380 |

> ⚠️ 연구/교육 목적입니다. 실제 투자에 단독 사용 금지.